In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
USER_DIR  = "Archived-users/Archived users"

In [7]:
import os
import pandas as pd
import numpy as np

TAPPY_DIR="Archived-Data/Tappy Data"
rows=[]
for fname in os.listdir(TAPPY_DIR):
    if not fname.endswith(".txt"):
        continue

    path=os.path.join(TAPPY_DIR,fname)
    if os.path.getsize(path)==0:
        continue

    try:
        df=pd.read_csv(path,sep="\t",header=None)
    except Exception:
        continue

    df=df.dropna(axis=1,how="all")
    if df.shape[1]<8:
        continue

    df.columns=[
        "UserKey","Date","Timestamp","Hand",
        "HoldTime","Direction","LatencyTime","FlightTime"
    ]

    num_cols=["HoldTime","LatencyTime","FlightTime"]
    df[num_cols]=df[num_cols].apply(pd.to_numeric,errors="coerce")
    df=df.dropna(subset=num_cols)
    if df.empty:
        continue
    user_id=df["UserKey"].iloc[0]
    left=df[df["Hand"]=="L"]
    right=df[df["Hand"]=="R"]

    def mean_safe(series):
        return series.mean() if not series.empty else np.nan
    mean_hold=df["HoldTime"].mean()
    std_hold=df["HoldTime"].std()
    mean_lat=df["LatencyTime"].mean()
    std_lat=df["LatencyTime"].std()
    mean_flt=df["FlightTime"].mean()
    std_flt=df["FlightTime"].std()

    # Left / Right means
    l_hold=mean_safe(left["HoldTime"])
    r_hold=mean_safe(right["HoldTime"])
    l_lat=mean_safe(left["LatencyTime"])
    r_lat=mean_safe(right["LatencyTime"])
    l_flt=mean_safe(left["FlightTime"])
    r_flt=mean_safe(right["FlightTime"])

    hold_asym=abs(l_hold-r_hold) if pd.notna(l_hold) and pd.notna(r_hold) else np.nan
    lat_asym=abs(l_lat-r_lat ) if pd.notna(l_lat ) and pd.notna(r_lat ) else np.nan
    flt_asym=abs(l_flt-r_flt ) if pd.notna(l_flt ) and pd.notna(r_flt ) else np.nan

    rows.append({
        "user_id":user_id,
        "mean_hold":mean_hold,
        "std_hold":std_hold,
        "mean_latency":mean_lat,
        "std_latency":std_lat,
        "mean_flight":mean_flt,
        "std_flight":std_flt,

        "l_hold":l_hold,
        "r_hold":r_hold,
        "l_latency":l_lat,
        "r_latency":r_lat,
        "l_flight":l_flt,
        "r_flight":r_flt,

        #KEY FEATURES
        "hold_asym":hold_asym,
        "lat_asym":lat_asym,
        "flight_asym":flt_asym
    })

#Session-level→user-level aggregation
features_df=pd.DataFrame(rows)
features_df=features_df.groupby("user_id",as_index=False).mean()

print("Feature shape:",features_df.shape)
features_df.head()


C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_34752\2014236287.py:16: DtypeWarning: Columns (4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path,sep="\t",header=None)
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_34752\2014236287.py:16: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path,sep="\t",header=None)
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_34752\2014236287.py:16: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path,sep="\t",header=None)
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_34752\2014236287.py:16: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path,sep="\t",header=None)
C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_34752\2014236287.py:16: DtypeWarning: Columns (7) have mixed types. Specify dtype opt

Feature shape: (266, 16)


,user_id,mean_hold,std_hold,mean_latency,std_latency,mean_flight,std_flight,l_hold,r_hold,l_latency,r_latency,l_flight,r_flight,hold_asym,lat_asym,flight_asym
0,0EA27ICBLF,88.080903,25.365159,284.668525,110.831540,202.450642,107.954312,78.719465,80.115836,321.837997,276.463879,235.803057,202.569935,1.396371,45.374118,33.233122
1,0NMZ9JBBHI,96.949927,35.029892,270.747253,97.381526,191.450380,94.886219,94.219589,97.300483,275.054185,258.986181,192.799009,189.581638,7.598402,16.068004,4.543894
2,0QAZFRHQHW,103.239969,32.693588,408.650526,172.730590,307.158495,168.857084,99.606155,101.929223,426.171897,402.454625,322.095736,301.536209,2.323068,23.717272,20.559528
3,1BMKGIOSHF,110.408824,30.431168,348.464706,83.284328,211.285294,94.699240,118.165000,102.161538,363.680000,319.107692,224.805000,184.200000,16.003462,44.572308,40.605000
4,1HOEBIGASW,65.568254,11.910890,463.917460,199.080825,396.244444,199.148889,66.280645,65.036667,456.151613,485.160000,386.722581,417.916667,1.243978,29.008387,31.194086


In [8]:
print("User-level feature shape:",features_df.shape)
features_df.head()


User-level feature shape: (266, 16)


,user_id,mean_hold,std_hold,mean_latency,std_latency,mean_flight,std_flight,l_hold,r_hold,l_latency,r_latency,l_flight,r_flight,hold_asym,lat_asym,flight_asym
0,0EA27ICBLF,88.080903,25.365159,284.668525,110.831540,202.450642,107.954312,78.719465,80.115836,321.837997,276.463879,235.803057,202.569935,1.396371,45.374118,33.233122
1,0NMZ9JBBHI,96.949927,35.029892,270.747253,97.381526,191.450380,94.886219,94.219589,97.300483,275.054185,258.986181,192.799009,189.581638,7.598402,16.068004,4.543894
2,0QAZFRHQHW,103.239969,32.693588,408.650526,172.730590,307.158495,168.857084,99.606155,101.929223,426.171897,402.454625,322.095736,301.536209,2.323068,23.717272,20.559528
3,1BMKGIOSHF,110.408824,30.431168,348.464706,83.284328,211.285294,94.699240,118.165000,102.161538,363.680000,319.107692,224.805000,184.200000,16.003462,44.572308,40.605000
4,1HOEBIGASW,65.568254,11.910890,463.917460,199.080825,396.244444,199.148889,66.280645,65.036667,456.151613,485.160000,386.722581,417.916667,1.243978,29.008387,31.194086


In [9]:
labels=[]

for fname in os.listdir(USER_DIR):
    if not fname.endswith(".txt"):
        continue
    user_id=fname.replace("User_","").replace(".txt","")
    path=os.path.join(USER_DIR,fname)

    label=None
    with open(path,"r",encoding="utf-8",errors="ignore") as f:
        for line in f:
            if line.lower().startswith("parkinsons"):
                label=1 if "true" in line.lower() else 0
                break

    if label is not None:
        labels.append({
            "user_id":user_id,
            "label":label
        })

labels_df=pd.DataFrame(labels)

print("Labels shape:",labels_df.shape)
labels_df.head()


Labels shape: (227, 2)


,user_id,label
0,0EA27ICBLF,1
1,0QAZFRHQHW,0
2,0WTDIGPSBZ,0
3,1HOEBIGASW,0
4,1WMVCCU4RH,1


In [10]:
final_df=features_df.merge(labels_df,on="user_id",how="inner")

print("Final dataset shape:",final_df.shape)
print(final_df.head())
print("\nLabel distribution:")
print(final_df["label"].value_counts())


Final dataset shape: (217, 17)
      user_id   mean_hold   std_hold  mean_latency  std_latency  mean_flight  \
0  0EA27ICBLF   88.080903  25.365159    284.668525   110.831540   202.450642   
1  0QAZFRHQHW  103.239969  32.693588    408.650526   172.730590   307.158495   
2  1HOEBIGASW   65.568254  11.910890    463.917460   199.080825   396.244444   
3  1XNJCXS3EY  123.793469  47.127055    325.569638    94.386097   208.651721   
4  2JTCBKUP8T   92.753020  30.188329    328.181562   163.191715   293.011413   

   std_flight      l_hold      r_hold   l_latency   r_latency    l_flight  \
0  107.954312   78.719465   80.115836  321.837997  276.463879  235.803057   
1  168.857084   99.606155  101.929223  426.171897  402.454625  322.095736   
2  199.148889   66.280645   65.036667  456.151613  485.160000  386.722581   
3   90.130824  153.702407  105.622423  328.312910  319.616275  226.013786   
4  155.774337   91.288092   93.390106  303.094524  342.548563  272.019421   

     r_flight  hold_asym 

In [11]:
print("Missing values per column:")
print(final_df.isna().sum())

print("\nColumns:")
print(final_df.columns)


Missing values per column:
user_id         0
mean_hold       0
std_hold        1
mean_latency    0
std_latency     1
mean_flight     0
std_flight      1
l_hold          3
r_hold          3
l_latency       3
r_latency       3
l_flight        3
r_flight        3
hold_asym       5
lat_asym        5
flight_asym     5
label           0
dtype: int64

Columns:
Index(['user_id', 'mean_hold', 'std_hold', 'mean_latency', 'std_latency',
       'mean_flight', 'std_flight', 'l_hold', 'r_hold', 'l_latency',
       'r_latency', 'l_flight', 'r_flight', 'hold_asym', 'lat_asym',
       'flight_asym', 'label'],
      dtype='object')


In [12]:
# Check missing values
print(final_df.isna().sum())
# Impute ALL numeric NaNs with column mean
numeric_cols=[c for c in final_df.columns if c not in ["label"] and final_df[c].dtype!="object"]


user_id         0
mean_hold       0
std_hold        1
mean_latency    0
std_latency     1
mean_flight     0
std_flight      1
l_hold          3
r_hold          3
l_latency       3
r_latency       3
l_flight        3
r_flight        3
hold_asym       5
lat_asym        5
flight_asym     5
label           0
dtype: int64


Group aware cross-validation(research oriented eval)


In [17]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, recall_score
import numpy as np
import pandas as pd

X = final_df.drop(columns=["user_id","label"])
y = final_df["label"]
groups = final_df["user_id"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]),
    "SVM(RBF)": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, class_weight="balanced"))
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model",RandomForestClassifier(
            n_estimators=400,
            random_state=42,
            class_weight="balanced"
        ))
    ]),
    "GradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", GradientBoostingClassifier(random_state=42))
    ])
}

results = []

for name, model in models.items():
    roc_scores, recall_scores, rec_pd_scores, rec_control_scores = [], [], [], []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)

        roc = roc_auc_score(y_test, y_prob)
        rec = recall_score(y_test, y_pred)
        rec_pd = recall_score(y_test, y_pred, pos_label=1)
        rec_control = recall_score(y_test, y_pred, pos_label=0)

        roc_scores.append(roc)
        recall_scores.append(rec)
        rec_pd_scores.append(rec_pd)
        rec_control_scores.append(rec_control)

        print(f"{name} | Fold {fold+1}: ROC-AUC={roc:.3f}, Recall={rec:.3f}, Recall_PD={rec_pd:.3f}, Recall_Control={rec_control:.3f}")

    results.append({
        "Model": name,
        "ROC-AUC(mean)": np.mean(roc_scores),
        "ROC-AUC(std)": np.std(roc_scores),
        "Recall(mean)": np.mean(recall_scores),
        "Recall(std)": np.std(recall_scores),
        "Recall_PD(mean)": np.mean(rec_pd_scores),
        "Recall_Control(mean)": np.mean(rec_control_scores)
    })

print("\n===== FINAL GROUP-CV COMPARISON =====")
results_df = pd.DataFrame(results).sort_values(by="ROC-AUC(mean)", ascending=False)
print(results_df.to_string(index=False))


LogisticRegression | Fold 1: ROC-AUC=0.364, Recall=0.692, Recall_PD=0.692, Recall_Control=0.200
LogisticRegression | Fold 2: ROC-AUC=0.524, Recall=0.484, Recall_PD=0.484, Recall_Control=0.615
LogisticRegression | Fold 3: ROC-AUC=0.350, Recall=0.500, Recall_PD=0.500, Recall_Control=0.200
LogisticRegression | Fold 4: ROC-AUC=0.533, Recall=0.618, Recall_PD=0.618, Recall_Control=0.444
LogisticRegression | Fold 5: ROC-AUC=0.615, Recall=0.633, Recall_PD=0.633, Recall_Control=0.769
SVM(RBF) | Fold 1: ROC-AUC=0.615, Recall=0.538, Recall_PD=0.538, Recall_Control=0.400
SVM(RBF) | Fold 2: ROC-AUC=0.603, Recall=0.484, Recall_PD=0.484, Recall_Control=0.385
SVM(RBF) | Fold 3: ROC-AUC=0.476, Recall=0.857, Recall_PD=0.857, Recall_Control=0.200
SVM(RBF) | Fold 4: ROC-AUC=0.444, Recall=0.529, Recall_PD=0.529, Recall_Control=0.667
SVM(RBF) | Fold 5: ROC-AUC=0.601, Recall=0.467, Recall_PD=0.467, Recall_Control=0.385
RandomForest | Fold 1: ROC-AUC=0.531, Recall=0.846, Recall_PD=0.846, Recall_Control=0.000


In [19]:
import joblib

X=final_df.drop(columns=["user_id","label"]).copy()
y=final_df["label"].copy()
final_model = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("model",GradientBoostingClassifier(random_state=42))
])

final_model.fit(X, y)

print("Final model trained on full dataset.")

Final model trained on full dataset.


In [20]:
MODEL_PATH="parkinson_keystroke_model.pkl"
joblib.dump(final_model,MODEL_PATH)
print(f" Model saved as: {MODEL_PATH}")

 Model saved as: parkinson_keystroke_model.pkl


In [21]:
FEATURES_PATH="features_columns.pkl"
joblib.dump(list(X.columns),FEATURES_PATH)

print(f"Feature columns saved as:{FEATURES_PATH}")
print("Feature count:",len(X.columns))


Feature columns saved as:features_columns.pkl
Feature count: 15
